# E5 BBQ explanation ablations

Run after all full CAB/BBQ cells finish and after this notebook's commit is pushed. Use NVIDIA L4. The four 60-item cells checkpoint after each item. If interrupted, repeat setup and the interrupted cell on the same commit. This notebook measures local section ablations, global summaries, and stability across the three cyclic answer orders. Token-level faithfulness and paraphrase stability remain pending.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys
drive.mount('/content/drive')
SHARED = Path('/content/drive/MyDrive/CMPE253-Project').resolve()
assert SHARED.is_dir()
REPO = Path('/content/cmpe253-llm-demographic-bias-audit')
URL = 'https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', URL, str(REPO)], check=True)
assert not subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['make', 'setup-colab'], cwd=REPO, check=True)
GPU = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
assert GPU == 'NVIDIA L4', GPU
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('Ready:', GPU, COMMIT)


In [ ]:
subprocess.run(['make', 'ai-review-gate', f'ROOT={SHARED}'], cwd=REPO, check=True)
subprocess.run(['make', 'report-bbq', f'ROOT={SHARED}'], cwd=REPO, check=True)
subprocess.run(['make', 'freeze-explain-sample', f'ROOT={SHARED}'], cwd=REPO, check=True)
print('Full BBQ report and frozen 60-item sample verified')


## M1 baseline and fairness instruction


In [ ]:
for run_id in ('EXPLAIN-M1-B0', 'EXPLAIN-M1-F1'):
    print('Starting', run_id, flush=True)
    subprocess.run(['make', 'run-explain', f'ROOT={SHARED}', f'RUN_ID={run_id}'], cwd=REPO, check=True)
    receipt = json.loads((SHARED / 'manifests/explain' / f'{run_id}.json').read_text())
    assert receipt['status'] == 'complete' and receipt['items'] == 60
    print('Completed', run_id, flush=True)


## M2 baseline and fairness instruction


In [ ]:
for run_id in ('EXPLAIN-M2-B0', 'EXPLAIN-M2-F1'):
    print('Starting', run_id, flush=True)
    subprocess.run(['make', 'run-explain', f'ROOT={SHARED}', f'RUN_ID={run_id}'], cwd=REPO, check=True)
    receipt = json.loads((SHARED / 'manifests/explain' / f'{run_id}.json').read_text())
    assert receipt['status'] == 'complete' and receipt['items'] == 60
    print('Completed', run_id, flush=True)


In [ ]:
subprocess.run(['make', 'report-explain', f'ROOT={SHARED}'], cwd=REPO, check=True)
path = SHARED / 'aggregate_results/explain_summary.json'
summary = json.loads(path.read_text())
assert summary['rows'] == 240 and len(summary['cells']) == 60
assert len(summary['answer_order_stability']) == 36
print('Verified explanation summary:', path)
print('Exact identity markers missing from', sum(cell['not_present_items'] for cell in summary['cells'] if cell['feature'] == 'identity_marker'), 'of 240 items')
